# 09 · Inferencia: del modelo final a una función que cualquiera puede usar

> Rehecho en `specs/007-integracion-main` (2026-11-06) sobre el **modelo final de la rama**: LightGBM con las 5 variables nuevas,
> sin `Policy_Sales_Channel` y con `Gender`. Guía: `material-del-curso/kit/guia-validacion-registro-y-app.md`, secciones 6 a 8.

Un modelo entrenado no es un producto. Un producto recibe datos **sucios o incompletos de una persona real** y responde con algo útil
o con un **error claro**. Esta etapa construye ese puente:

1. **Exportar** el modelo final a un archivo que viaja con la app (`models/model.joblib`).
2. Escribir el **esquema de entrada** (`models/input_schema.json`): qué columnas, qué rangos son posibles, qué categorías.
3. Probar la predicción con datos buenos, incompletos e **imposibles**.
4. Comprobar que el archivo exportado **es** el modelo final (mismas probabilidades).

**La prueba no se abre en este notebook.** Todo se hace con clientes de entrenamiento.

## 0. Datos y configuración

In [1]:
import json
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import joblib
import numpy as np
import pandas as pd

from proyecto.config import COST_MATRIX, INPUT_VALID_RANGE, TARGET
from proyecto.inference.serving import NUMERIC_RAW, RAW_COLUMNS, CATEGORY_TRANSLATION, ServingModel, build_defaults, to_raw

train = pd.read_parquet(ROOT / "data" / "processed" / "train.parquet")
raw_train = to_raw(train)                       # lo que escribiría una persona: "Yes"/"No", "< 1 Year", etc.

NUMERICAS = list(NUMERIC_RAW)                   # Age, Annual_Premium, Region_Code, Vintage
CATEGORICAS = list(CATEGORY_TRANSLATION)        # Gender, Driving_License, Previously_Insured, Vehicle_Age, Vehicle_Damage
CLASES = ["no interesado", "interesado"]        # en el orden de predict_proba (0, 1)
MATRIZ_DE_COSTOS = COST_MATRIX                  # filas = real, columnas = decisión; perder un interesado cuesta 5 llamadas inútiles
# Lo POSIBLE en el mundo real (decisión del grupo, con su motivo en config.py). Fuera de esto la predicción se niega.
RANGOS_POSIBLES = INPUT_VALID_RANGE

assert sorted(NUMERICAS + CATEGORICAS) == sorted(RAW_COLUMNS)
print(f"entrenamiento: {len(train):,} clientes; el formulario pedirá {len(RAW_COLUMNS)} datos: {RAW_COLUMNS}")

entrenamiento: 304,238 clientes; el formulario pedirá 9 datos: ['Age', 'Annual_Premium', 'Region_Code', 'Vintage', 'Gender', 'Driving_License', 'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage']


## 1. Exportar el modelo final

Se usa la copia local de la etapa 7 (`models/final_pipeline.joblib`). **No** se lee del registro de Databricks: hoy el alias
`champion` de `workspace.default.venta_cruzada_seguros` todavía apunta al modelo anterior de `main` (versión 1). Cuando se ejecute
la tarea T009 de `specs/006-registro-modelo`, `champion` pasará a este mismo modelo.

`ServingModel` envuelve el modelo con la traducción "lo que escribe una persona → columnas limpias".

In [2]:
pipeline = joblib.load(ROOT / "models" / "final_pipeline.joblib")
defaults = build_defaults(raw_train)            # valores neutros para los "no sé", solo del entrenamiento
modelo = ServingModel(pipeline, defaults)
joblib.dump(modelo, ROOT / "models" / "model.joblib")
print("modelo exportado a models/model.joblib desde: models/final_pipeline.joblib (etapa 7)")
print("valores usados cuando alguien marca «no sé»:", defaults)

modelo exportado a models/model.joblib desde: models/final_pipeline.joblib (etapa 7)
valores usados cuando alguien marca «no sé»: {'Age': 36.0, 'Annual_Premium': 31725.0, 'Region_Code': 28, 'Vintage': 154.0, 'Gender': 'Male', 'Driving_License': 'Yes', 'Previously_Insured': 'No', 'Vehicle_Age': '1-2 Year', 'Vehicle_Damage': 'Yes'}


## 2. El esquema de entrada

La app y la función de predicción necesitan saber qué se puede preguntar. Se calcula una vez con el **entrenamiento** y se guarda en
`models/input_schema.json`, que viaja con el modelo.

La decisión es la de **menor costo esperado** con la matriz `[[0, 1], [5, 0]]`: equivale a llamar si P(interesado) > 1/6, la misma
regla de la etapa 7.

In [3]:
from proyecto.inference.schema import build_schema, save_schema

LABELS = {"Age": "Edad (años)", "Annual_Premium": "Prima anual del seguro de salud", "Region_Code": "Código de región",
          "Vintage": "Días como cliente de la aseguradora", "Gender": "Género",
          "Driving_License": "¿Tiene licencia de conducir?", "Previously_Insured": "¿Ya tiene seguro de vehículo?",
          "Vehicle_Age": "Antigüedad del vehículo", "Vehicle_Damage": "¿El vehículo ha tenido daños?"}
schema = build_schema(raw_train, NUMERICAS, CATEGORICAS, CLASES, cost_matrix=MATRIZ_DE_COSTOS, valid_range=RANGOS_POSIBLES,
                      labels=LABELS, review_threshold=0.0)
schema["app_title"] = "¿A qué clientes conviene llamar? Venta cruzada de seguro de vehículo"
schema["decision_labels"] = {"interesado": "LLAMAR", "no interesado": "NO LLAMAR"}
schema["responsible_use"] = (
    "- Esta herramienta **ordena a quién llamar primero**; no decide si se vende o se niega un seguro.\n"
    "- Usa la **edad y el género** del cliente. Legal debe aprobar su uso; los resultados por edad y género están en la ficha del modelo.\n"
    "- Entrenada con datos históricos de una sola aseguradora: no la uses con otra población sin validarla de nuevo.\n"
    "- El costo asume que perder un comprador cuesta 5 veces una llamada inútil (hipótesis por confirmar).\n"
    "- Revisa los **límites del modelo** en `models/ficha_modelo.md`."
)
save_schema(schema, ROOT / "models" / "input_schema.json")
print(f"esquema guardado: {len(NUMERICAS)} numéricas, {len(CATEGORICAS)} categóricas, clases {CLASES}")
print("categorías conocidas:", schema["categories"])

esquema guardado: 4 numéricas, 5 categóricas, clases ['no interesado', 'interesado']


categorías conocidas: {'Gender': ['Female', 'Male'], 'Driving_License': ['No', 'Yes'], 'Previously_Insured': ['No', 'Yes'], 'Vehicle_Age': ['1-2 Year', '< 1 Year', '> 2 Years'], 'Vehicle_Damage': ['No', 'Yes']}


## 3. Probar la predicción

Casos que importan: una fila normal, la misma con otro perfil, una **incompleta** (el usuario marcó "no sé"), una con un dato
**imposible** y una con una categoría desconocida. Se espera: las tres primeras responden; las dos últimas **se niegan y explican por qué**.

In [4]:
from proyecto.inference.predict import InputValidationError, predict_with_scores

modelo = joblib.load(ROOT / "models" / "model.joblib")
normal = pd.DataFrame([{"Age": 34, "Annual_Premium": 31000.0, "Region_Code": 28, "Vintage": 150, "Gender": "Male",
                        "Driving_License": "Yes", "Previously_Insured": "No", "Vehicle_Age": "1-2 Year", "Vehicle_Damage": "Yes"}])
print("— fila normal (hombre de 34 años, sin seguro previo, con daño) —")
print(predict_with_scores(normal, modelo, schema).round(3).to_string())

contraste = normal.assign(Vehicle_Damage="No", Previously_Insured="Yes")
print("\n— el mismo cliente con seguro previo y sin daño —")
print(predict_with_scores(contraste, modelo, schema).round(3).to_string())

incompleta = normal.assign(Age=np.nan, Gender=np.nan)
print("\n— con la edad y el género en «no sé» —")
print(predict_with_scores(incompleta, modelo, schema).round(3).to_string())

imposible = normal.assign(Age=150)
try:
    predict_with_scores(imposible, modelo, schema)
    raise AssertionError("debía negarse a predecir con una edad de 150 años")
except InputValidationError as error:
    print("\n— dato imposible (edad 150) —\nError claro:", error)

categoria_rara = normal.assign(Vehicle_Age="3 años")
try:
    predict_with_scores(categoria_rara, modelo, schema)
    raise AssertionError("debía rechazar una categoría desconocida")
except InputValidationError as error:
    print("\n— categoría desconocida —\nSe rechaza con:", error)

— fila normal (hombre de 34 años, sin seguro previo, con daño) —


     decision  proba_no interesado  proba_interesado    most_likely  expected_cost  needs_review
0  interesado                0.609             0.391  no interesado          0.609          True

— el mismo cliente con seguro previo y sin daño —
        decision  proba_no interesado  proba_interesado    most_likely  expected_cost  needs_review
0  no interesado                0.997             0.003  no interesado          0.013         False

— con la edad y el género en «no sé» —
     decision  proba_no interesado  proba_interesado    most_likely  expected_cost  needs_review
0  interesado                0.633             0.367  no interesado          0.633          True

— dato imposible (edad 150) —
Error claro: Age: debe estar entre 18 y 100

— categoría desconocida —
Se rechaza con: Vehicle_Age: valor desconocido ['3 años']; se esperaba uno de ['1-2 Year', '< 1 Year', '> 2 Years']


## 4. Comprobar que el archivo exportado es el modelo final

`models/model.joblib` (con la traducción) debe dar **las mismas probabilidades** que `models/final_pipeline.joblib` (el modelo que
se evaluó en la etapa 7). Se comprueba con 200 clientes de **entrenamiento**: la prueba no se vuelve a abrir. Esta comprobación no
cambia nada del modelo ni decide nada.

In [5]:
muestra = train.drop(columns=TARGET).sample(200, random_state=42)
diferencia = float(np.abs(modelo.predict_proba(to_raw(muestra)) - pipeline.predict_proba(muestra)).max())
print(f"diferencia máxima de probabilidades en 200 clientes de entrenamiento: {diferencia:.3g}")
assert diferencia <= 1e-9, "el modelo exportado no es el modelo final"
print("el modelo exportado es el modelo final")

diferencia máxima de probabilidades en 200 clientes de entrenamiento: 0
el modelo exportado es el modelo final


## 5. Siguiente: la app

Con `models/model.joblib` y `models/input_schema.json` en su sitio, la app funciona:

```bash
streamlit run app/streamlit_app.py
```

Comprueba que: el formulario pide 9 datos (sin canal de venta), una predicción normal responde con LLAMAR o NO LLAMAR,
**"No sé" no rompe nada** y una edad de 150 muestra un error claro.